## Preprocessing edf format signals and hypnograms

1. Linking corresponding recording and hypnogram
2. Load data and convert sleep stages to numeric labels
3. Cut recordings before and after sleep
4. Convert to mne.Raw objects and save as .fif files

In [22]:
# imports
import os
import re
import mne
import numpy as np
from tqdm import tqdm

mne.set_log_level("WARNING")

In [23]:
# define path to the data
# ADAPT THIS PATH TO WHERE YOU HAVE THE DATA
data_path = "."

In [24]:
folder_path = "sleep-edf-database"
sc = "sleep-cassette"
st = "sleep-telemetry"

sc_path = os.path.join(data_path, folder_path, sc)
st_path = os.path.join(data_path, folder_path, st)

In [25]:
# Function to extract subject ID as an integer from the edf file path
# e.g. for SC4001E0-PSG.edf extract 0 (participant 0)
def extract_subject_id(file_path):
    # extract subject number from file name
    pattern = r'SC4(\d{2})\d'
    match = re.search(pattern, file_path)
    if match:
        return int(match.group(1)) # return subject ID as integer (0 would be intire matched string)
    return None


# get lists of psg file paths and corresponding hypnogram file paths
def get_data_paths(directory):
    all_files = os.listdir(directory)

    psg_files = [f for f in all_files if f.endswith('PSG.edf')]
    hypnogram_files = [f for f in all_files if f.endswith('Hypnogram.edf')]
    
    sorted_psg_paths = []
    sorted_hypnogram_paths = []

    # pair psg and hypnogram files based on participant and night name -> there is always a pair of PSG and Hypnogram files
    # e.g. for SC4001E0-PSG.edf extract identifier SC4001
    for psg_file in psg_files:
        base_name = psg_file.split('-')[0]
        identifier = base_name[:-2]
        subject_id = extract_subject_id(psg_file)
        for hypnogram_file in hypnogram_files:
            if hypnogram_file.startswith(identifier):
                sorted_psg_paths.append(os.path.join(directory, psg_file))
                sorted_hypnogram_paths.append(os.path.join(directory, hypnogram_file))

    # returns 2 lists of paths (1 for PSG 1 for Hypnograms) sorted in the same order
    return sorted_psg_paths, sorted_hypnogram_paths


In [26]:
sc_psg_paths, sc_hypnogram_paths = get_data_paths(sc_path)
st_psg_paths, st_hypnogram_paths = get_data_paths(st_path)

In [27]:
# convert sleep stage (string) to numeric stage
def map_sleep_stage_to_label(stage):
    # map based on AASM standards (no distinction between stage 3 and 4)
    stage_mapping = {
    'Sleep stage W': 0,
    'Sleep stage 1': 1,
    'Sleep stage 2': 2,
    'Sleep stage 3': 3,
    'Sleep stage 4': 3,
    'Sleep stage R': 4,
    }
    return stage_mapping.get(stage, -1) # -1 is Sleep Stage ? or Movement time

# load data from edf files
def load_data(psg_path, hypnogram_path, epoch_duration=30.0):
    
    raw = mne.io.read_raw_edf(psg_path, preload=False, verbose='ERROR')

    psg_data = raw.get_data()

    hypnogram = mne.read_annotations(hypnogram_path)

    sfreq = raw.info['sfreq'] # is 100 Hz

    ch_names = raw.info['ch_names']
    ch_types = raw.get_channel_types()

    # Calculate total epochs based on signal duration
    samples_per_epoch = int(sfreq * epoch_duration)
    total_epochs = psg_data.shape[1] // samples_per_epoch
    sleep_stage_labels = np.full(total_epochs, -1, dtype=np.int32)  # -1 for unknown stages

    # Fill labels based on annotation onsets and durations
    for row in hypnogram:
        stage_label = map_sleep_stage_to_label(row['description'])
        start_epoch = int(row['onset'] // epoch_duration)
        end_epoch = start_epoch + int(row['duration'] // epoch_duration)
        if end_epoch > total_epochs:
            end_epoch = total_epochs
        sleep_stage_labels[start_epoch:end_epoch] = stage_label

    return (psg_data, sleep_stage_labels, sfreq, ch_names, ch_types)

In [ ]:
# removing wake stages before and after sleep
def remove_wake(signals, labels, wake_stage_index=0, sampling_rate=100, epoch_duration=30.0):
    samples_per_epoch = int(sampling_rate * epoch_duration)

    # 1. Find indices of all non-wake stages
    non_wake_indices = np.where((labels != wake_stage_index) & (labels != -1))[0]

    if len(non_wake_indices) == 0:
        raise ValueError("No non-wake stages found in the data.")
    
    # 2. Break sleep into blocks separated by > 1 hour of WAKE (120 epochs)
    gap_threshold_1_hr = 120
    sleep_gaps = np.diff(non_wake_indices)
    split_indices = np.where(sleep_gaps > gap_threshold_1_hr)[0] + 1
    blocks = np.split(non_wake_indices, split_indices)
    
    # 3. Destroy "microsleep bridges" by dropping blocks with < 30 mins of total sleep
    # 30 mins = 60 epochs. This cuts out short afternoon naps 
    valid_blocks = [b for b in blocks if len(b) >= 60]
    
    if len(valid_blocks) == 0:
        # Fallback in extreme cases
        valid_blocks = [max(blocks, key=len)]
        
    # 4. Find the "Core Night" (the block with the absolute most sleep)
    core_idx = np.argmax([len(b) for b in valid_blocks])
    
    # 5. Expand left and right to reconnect middle-of-the-night insomnia gaps
    # We absorb adjacent blocks if the gap is < 3 hours (360 epochs)
    merge_gap = 360
    
    start_idx = core_idx
    while start_idx > 0:
        # Calculate gap between current block and the one to its left
        gap = valid_blocks[start_idx][0] - valid_blocks[start_idx-1][-1]
        if gap < merge_gap:
            start_idx -= 1
        else:
            break  # Stop if the gap is too large (e.g., an afternoon nap 5 hours ago)
            
    end_idx = core_idx
    while end_idx < len(valid_blocks) - 1:
        # Calculate gap between current block and the one to its right
        gap = valid_blocks[end_idx+1][0] - valid_blocks[end_idx][-1]
        if gap < merge_gap:
            end_idx += 1
        else:
            break
            
    # 6. Extract the true boundaries of the main night
    first_non_wake_epoch = valid_blocks[start_idx][0]
    last_non_wake_epoch = valid_blocks[end_idx][-1]

    # 7. Add context (Keep 15 minutes / 30 epochs of WAKE before and after)
    context_epochs = 30
    start_epoch = max(0, first_non_wake_epoch - context_epochs)
    end_epoch = min(len(labels), last_non_wake_epoch + context_epochs + 1)

    # 8. Trim labels and signals
    trimmed_labels = labels[start_epoch:end_epoch]
    
    start_index = start_epoch * samples_per_epoch
    end_index = end_epoch * samples_per_epoch
    trimmed_signals = signals[:, start_index:end_index]

    return trimmed_signals, trimmed_labels

In [29]:
def hypnogram_to_annotations(labels, epoch_duration=30.0):
    onsets = np.arange(len(labels)) * epoch_duration
    durations = np.full(len(labels), epoch_duration)
    descriptions = labels.astype(str)
    
    return mne.Annotations(onsets, durations, descriptions)

In [30]:
for psg_path, hypnogram_path in tqdm(zip(sc_psg_paths, sc_hypnogram_paths), desc="Processing sleep-cassette data", total=len(sc_psg_paths)):
    signal, label, sfreq, ch_names, ch_types = load_data(psg_path, hypnogram_path)
    trimmed_sig, trimmed_labels = remove_wake(signal, label)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw_new = mne.io.RawArray(trimmed_sig, info)
    annot = hypnogram_to_annotations(trimmed_labels)
    raw_new.set_annotations(annot)
    out_dir = sc_path + "/preprocessed/"
    os.makedirs(out_dir, exist_ok=True)
    fname = os.path.basename(psg_path).replace("PSG.edf", "raw.fif")
    out_path = os.path.join(out_dir, fname)
    raw_new.save(out_path, overwrite=True)


for psg_path, hypnogram_path in tqdm(zip(st_psg_paths, st_hypnogram_paths), desc="Processing sleep-telemetry data", total=len(st_psg_paths)):
    signal, label, sfreq, ch_names, ch_types = load_data(psg_path, hypnogram_path)
    trimmed_sig, trimmed_labels = remove_wake(signal, label)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw_new = mne.io.RawArray(trimmed_sig, info)
    annot = hypnogram_to_annotations(trimmed_labels)
    raw_new.set_annotations(annot)
    out_dir = st_path + "/preprocessed/"
    os.makedirs(out_dir, exist_ok=True)
    fname = os.path.basename(psg_path).replace("PSG.edf", "raw.fif")
    out_path = os.path.join(out_dir, fname)
    raw_new.save(out_path, overwrite=True)


Processing sleep-telemetry data: 100%|██████████| 44/44 [06:29<00:00,  8.85s/it]
